In [13]:
import hashlib, json, copy
import pandas as pd

df = pd.read_csv("data/dashboard_incidents.csv")

def make_hash(entry, prev_hash):
    text = json.dumps(entry, sort_keys=True) + prev_hash
    return hashlib.sha256(text.encode()).hexdigest()

# build the chain
log, prev = [], "0" * 64
for _, row in df.iterrows():
    entry = json.loads(json.dumps(row.to_dict(), default=str))
    h = make_hash(entry, prev)
    log.append({"entry": entry, "prev_hash": prev, "hash": h})
    prev = h

def verify(log):
    prev = "0" * 64
    for i, rec in enumerate(log):
        if rec["prev_hash"] != prev or make_hash(rec["entry"], prev) != rec["hash"]:
            return False, i
        prev = rec["hash"]
    return True, None

print("Before tampering:", verify(log))

# save the REAL log first
json.dump(log, open("audit_log.json", "w"))

# tamper demo on a COPY only
demo = copy.deepcopy(log)
k = list(demo[5]["entry"].keys())[0]
demo[5]["entry"][k] = "TAMPERED"
print("After tampering (copy):", verify(demo))
print("Real log still valid:", verify(log))

Before tampering: (True, None)
After tampering (copy): (False, 5)
Real log still valid: (True, None)
